# Primary Model

## Problem Definition

**Question.** Which direction classifier best predicts `direction_label` in {-1, 1} without using future event outcomes?

**Role in the workflow.** Select and tune the primary direction model, generate development OOF predictions, and evaluate the sealed holdout once.

**Inputs.** Cleaned, weighted events, the split contract, and 53 event-start sentiment/fractional-price/technical features.

**Outputs.** Primary model artifact, candidate/tuning/importance tables, and OOF plus holdout side/probability/confidence predictions in `data/model_artifact/`.

**Why this method.** Weighted negative log loss rewards calibrated class probabilities needed by meta-labeling and bet sizing.

**Assumptions.** Only development selects the model; identifiers and all future outcomes are forbidden features; random state is 42.

**Handoff.** Primary OOF predictions to `meta_model.ipynb`, and fixed holdout predictions to final evaluation.


## Test Set Isolation and Development Exploration

- The upstream candidate split is reused before class balance, comparison, tuning, or importance is computed.
- All displayed selection evidence below is development-only.
- Bagging, random forest, AdaBoost, and gradient boosting are compared with random state 42 on five purged folds with a 1% embargo.
- Weighted log loss selects the family because calibrated direction probabilities are needed downstream; F1 and precision remain diagnostics rather than tie-breaking searches on the holdout.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score, log_loss, precision_score
from sklearn.ensemble import RandomForestClassifier

PROJECT_ROOT = Path.cwd().resolve().parents[1]

from src.strategy_modeling.cross_validation import PurgedKFold
from src.strategy_modeling.feature_importance import get_orthogonal_features
from src.strategy_modeling.model_workflow import (
    build_candidate_classifiers,
    candidate_parameter_grids,
    generate_oof_predictions,
    get_primary_feature_columns,
    score_binary_predictions,
)

RANDOM_STATE = 42
period = "2025-01-01_2025-12-31"
event_path = PROJECT_ROOT / f"data/research_data/events/aapl_news_modeling_prepared_{period}.parquet"
artifact_dir = PROJECT_ROOT / "data/model_artifact"
artifact_dir.mkdir(parents=True, exist_ok=True)

events = pd.read_parquet(event_path).sort_values("event_start", ignore_index=True)
split_manifest = pd.read_parquet(artifact_dir / "split_manifest.parquet")
development_starts = split_manifest.loc[split_manifest["partition"].eq("development"), "event_start"]
holdout_starts = split_manifest.loc[split_manifest["partition"].eq("holdout"), "event_start"]
development = events[events["event_start"].isin(development_starts)].copy()
holdout = events[events["event_start"].isin(holdout_starts)].copy()
feature_columns = get_primary_feature_columns(events)

development = development.set_index("event_start")
holdout = holdout.set_index("event_start")
X_development = development[feature_columns]
y_development = development["direction_label"].astype("int8")
w_development = development["sample_weight"].astype(float)
t1_development = development["event_end"]
cv = PurgedKFold(n_splits=5, t1=t1_development, pct_embargo=0.01)

candidates = build_candidate_classifiers(random_state=RANDOM_STATE, n_jobs=1)
comparison_rows = []
for name, estimator in candidates.items():
    predictions = generate_oof_predictions(estimator, X_development, y_development, w_development, cv, positive_label=1)
    scores = score_binary_predictions(
        y_development,
        predictions["prediction"],
        predictions["probability"],
        w_development,
        class_labels=[-1, 1],
        positive_label=1,
    )
    comparison_rows.append({
        "candidate": name,
        "weighted_log_loss": scores["log_loss"],
        "f1": scores["f1"],
        "precision": scores["precision"],
    })

comparison = pd.DataFrame(comparison_rows).set_index("candidate").sort_values(["weighted_log_loss", "f1"], ascending=[True, False])
selected_name = comparison.index[0]
display(comparison)


/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:919: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:925: RuntimeWarning: invalid value encountered in divide
  oob_decision_function = predictions / predictions.sum(axis=1)[:, np.newaxis]


/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:919: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:925: RuntimeWarning: invalid value encountered in divide
  oob_decision_function = predictions / predictions.sum(axis=1)[:, np.newaxis]
/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:919: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:925: RuntimeWarning: invalid value encountered in divide
  oob_decision_function

,weighted_log_loss,f1,precision
candidate,,,
random_forest,0.702833,0.599445,0.611474
adaboost,0.741780,0.482511,0.483281
bagging,0.754375,0.497766,0.500218
gradient_boosting,1.141570,0.515583,0.523464


## Purged Tuning and Development OOF Output

- Only the winning family is tuned on the same five purged folds.
- The resulting OOF predictions are produced by estimators that did not fit their prediction rows; their provenance is stored explicitly for meta-label validation.
- Only the selected family is searched over its three-value compact grid, using the same weighted log-loss objective and development folds.
- This limits computation and researcher degrees of freedom, while the retained OOF source and fold columns make the no-in-sample-prediction contract auditable.

In [2]:
tuning_rows = []
tuned_oof_by_configuration = {}
for configuration in candidate_parameter_grids()[selected_name]:
    estimator = clone(candidates[selected_name]).set_params(**configuration)
    predictions = generate_oof_predictions(estimator, X_development, y_development, w_development, cv, positive_label=1)
    key = repr(configuration)
    tuned_oof_by_configuration[key] = predictions
    scores = score_binary_predictions(
        y_development,
        predictions["prediction"],
        predictions["probability"],
        w_development,
        class_labels=[-1, 1],
        positive_label=1,
    )
    tuning_rows.append({
        "configuration": key,
        **configuration,
        "weighted_log_loss": scores["log_loss"],
        "f1": scores["f1"],
        "precision": scores["precision"],
    })

tuning = pd.DataFrame(tuning_rows).sort_values(["weighted_log_loss", "f1"], ascending=[True, False], ignore_index=True)
best_configuration = {key: tuning.loc[0, key] for key in candidate_parameter_grids()[selected_name][0]}
tuned_estimator = clone(candidates[selected_name]).set_params(**best_configuration)
primary_oof = tuned_oof_by_configuration[tuning.loc[0, "configuration"]]

primary_oof_output = development[["event_end", "raw_return", "direction_label", "sample_weight"]].copy()
primary_oof_output["partition"] = "development"
primary_oof_output["primary_side"] = primary_oof["prediction"].astype("int8")
primary_oof_output["primary_probability"] = primary_oof["probability"]
primary_oof_output["primary_probability_negative"] = 1.0 - primary_oof["probability"]
primary_oof_output["primary_probability_positive"] = primary_oof["probability"]
primary_oof_output["primary_class_probability"] = np.where(primary_oof_output["primary_side"].eq(1), primary_oof_output["primary_probability_positive"], primary_oof_output["primary_probability_negative"])
primary_oof_output["primary_confidence"] = np.maximum(primary_oof["probability"], 1.0 - primary_oof["probability"])
primary_oof_output["prediction_source"] = primary_oof["prediction_source"]
primary_oof_output["cv_fold"] = primary_oof["fold"]

display(tuning)
display(primary_oof_output.head())


,configuration,model__max_features,weighted_log_loss,f1,precision
0,{'model__max_features': 'sqrt'},sqrt,0.702833,0.599445,0.611474
1,{'model__max_features': 0.5},0.5,0.759457,0.450385,0.465814
2,{'model__max_features': 1.0},1.0,0.772318,0.426122,0.441960


,event_end,raw_return,direction_label,sample_weight,partition,primary_side,primary_probability,primary_probability_negative,primary_probability_positive,primary_class_probability,primary_confidence,prediction_source,cv_fold
event_start,,,,,,,,,,,,,
2025-01-13 14:30:01.329809+00:00,2025-01-13 14:34:17.401727+00:00,-0.006778,-1,4.044092,development,-1,0.450000,0.550000,0.450000,0.550000,0.550000,oof,0
2025-01-16 14:30:01.488226+00:00,2025-01-16 14:34:43.996829+00:00,-0.004820,-1,0.484717,development,1,0.591667,0.408333,0.591667,0.591667,0.591667,oof,0
2025-01-17 14:30:01.792073+00:00,2025-01-17 14:39:01.069235+00:00,-0.007032,-1,0.750442,development,-1,0.275000,0.725000,0.275000,0.725000,0.725000,oof,0
2025-01-17 17:20:53.298606+00:00,2025-01-21 14:30:00.854179+00:00,-0.025297,-1,1.049062,development,-1,0.341667,0.658333,0.341667,0.658333,0.658333,oof,0
2025-01-21 14:30:00.854179+00:00,2025-01-21 14:34:53.478136+00:00,-0.007594,-1,1.878389,development,-1,0.483333,0.516667,0.483333,0.516667,0.516667,oof,0


## Development Feature Importance and Error Analysis

- MDI, MDA, and SFI use development only.
- A separate random-forest diagnostic supplies comparable named-feature importances even if another family wins; orthogonal analysis reports redundancy without replacing the final feature schema.
- The diagnostic forest fixes 120 trees, balanced-subsample class weights, `sqrt` feature sampling, random state 42, and one worker so MDI, MDA, and SFI have a common reference estimator even when another family wins.
- MDA repeats each permutation five times to reduce single-shuffle noise, and the 95% orthogonal threshold diagnoses redundancy without replacing named features.

In [3]:
diagnostic_forest = RandomForestClassifier(
    n_estimators=120,
    class_weight="balanced_subsample",
    max_features="sqrt",
    n_jobs=1,
    random_state=RANDOM_STATE,
).fit(X_development, y_development, sample_weight=w_development)

mdi = pd.Series(diagnostic_forest.feature_importances_, index=feature_columns, name="mdi")
mda_result = permutation_importance(
    diagnostic_forest,
    X_development,
    y_development,
    scoring="neg_log_loss",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=1,
    sample_weight=w_development,
)
mda = pd.Series(mda_result.importances_mean, index=feature_columns, name="mda")

sfi_scores = {}
for feature in feature_columns:
    single_predictions = generate_oof_predictions(
        candidates["random_forest"],
        X_development[[feature]],
        y_development,
        w_development,
        cv,
        positive_label=1,
    )
    scores = score_binary_predictions(
        y_development,
        single_predictions["prediction"],
        single_predictions["probability"],
        w_development,
        class_labels=[-1, 1],
        positive_label=1,
    )
    sfi_scores[feature] = -scores["log_loss"]
sfi = pd.Series(sfi_scores, name="sfi_neg_log_loss")

importance = pd.concat([mdi, mda, sfi], axis=1).sort_values("mda", ascending=False)
orthogonal = get_orthogonal_features(X_development, var_thres=0.95)
orthogonal_summary = pd.DataFrame(
    {
        "component": orthogonal.columns,
        "label_correlation": [orthogonal[column].corr(y_development) for column in orthogonal.columns],
    }
)

importance.to_parquet(artifact_dir / "primary_feature_importance.parquet")
orthogonal_summary.to_parquet(artifact_dir / "primary_orthogonal_features.parquet", index=False)
display(importance.head(10))
display(orthogonal_summary.head())


,mdi,mda,sfi_neg_log_loss
Stochastic %D,0.043712,0.043978,-3.126625
Stochastic %K,0.029076,0.032828,-1.147831
Percentage Price Oscillator,0.028565,0.031672,-2.367254
Williams %R,0.030181,0.031558,-1.147831
Average True Range,0.036050,0.030580,-3.779821
True Range,0.027279,0.029611,-3.802409
TRIX,0.030989,0.028875,-1.909035
Detrended Price Oscillator,0.024968,0.028731,-2.061531
On-Balance Volume,0.032644,0.027072,-6.223195
Commodity Channel Index,0.029370,0.025849,-3.619146


,component,label_correlation
0,PC_1,-0.000612
1,PC_2,0.021498
2,PC_3,-0.031844
3,PC_4,-0.105901
4,PC_5,0.158147


## Final Fit and One-Time Holdout Evaluation

- The tuned family is fitted on all development rows, then the holdout is predicted once.
- No code below changes the selected family, features, or hyperparameters after seeing holdout metrics.
- The chosen family and configuration are refitted once on all development rows before any holdout metric is computed.
- The resulting model, feature order, boundary, and random state are saved together so the one-time evaluation is reproducible and cannot silently trigger another selection round.

In [4]:
final_primary = clone(tuned_estimator).fit(
    X_development,
    y_development,
    sample_weight=w_development.to_numpy(),
)
holdout_probability = final_primary.predict_proba(holdout[feature_columns])[:, list(final_primary.classes_).index(1)]
holdout_side = final_primary.predict(holdout[feature_columns]).astype("int8")

primary_holdout_output = holdout[["event_end", "raw_return", "direction_label", "sample_weight"]].copy()
primary_holdout_output["partition"] = "holdout"
primary_holdout_output["primary_side"] = holdout_side
primary_holdout_output["primary_probability"] = holdout_probability
primary_holdout_output["primary_probability_negative"] = 1.0 - holdout_probability
primary_holdout_output["primary_probability_positive"] = holdout_probability
primary_holdout_output["primary_class_probability"] = np.where(primary_holdout_output["primary_side"].eq(1), primary_holdout_output["primary_probability_positive"], primary_holdout_output["primary_probability_negative"])
primary_holdout_output["primary_confidence"] = np.maximum(holdout_probability, 1.0 - holdout_probability)
primary_holdout_output["prediction_source"] = "holdout"
primary_holdout_output["cv_fold"] = pd.NA

holdout_metrics = pd.Series(
    {
        "weighted_log_loss": log_loss(holdout["direction_label"], np.column_stack([1.0 - holdout_probability, holdout_probability]), labels=[-1, 1], sample_weight=holdout["sample_weight"]),
        "f1": f1_score(holdout["direction_label"], holdout_side, pos_label=1, sample_weight=holdout["sample_weight"]),
        "precision": precision_score(holdout["direction_label"], holdout_side, pos_label=1, sample_weight=holdout["sample_weight"], zero_division=0),
    },
    name="holdout",
)

primary_predictions = pd.concat([primary_oof_output, primary_holdout_output]).sort_index()
tuning_artifact = tuning.copy()
for column in tuning_artifact.columns:
    if column.startswith("model__") and tuning_artifact[column].dtype == "object":
        tuning_artifact[column] = tuning_artifact[column].astype(str)
primary_predictions.to_parquet(artifact_dir / "primary_predictions.parquet")
comparison.to_parquet(artifact_dir / "primary_candidate_metrics.parquet")
tuning_artifact.to_parquet(artifact_dir / "primary_tuning_metrics.parquet", index=False)
holdout_metrics.to_frame().to_parquet(artifact_dir / "primary_holdout_metrics.parquet")
joblib.dump(
    {
        "estimator": final_primary,
        "feature_columns": feature_columns,
        "selected_candidate": selected_name,
        "best_configuration": best_configuration,
        "holdout_boundary": holdout.index.min(),
        "random_state": RANDOM_STATE,
    },
    artifact_dir / "primary_model.joblib",
)

display(holdout_metrics.to_frame())
print(artifact_dir / "primary_model.joblib")


,holdout
weighted_log_loss,0.632087
f1,0.714774
precision,0.718047


/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/model_artifact/primary_model.joblib


## Results, Limitations, and Handoff

- Development OOF performance includes model-family and grid-selection uncertainty, and the one-year AAPL holdout is small.
- The stored holdout result is final even if it is worse than development; no retuning follows.
- The next notebook receives OOF-only development sides/confidences and fixed holdout predictions.
- No conclusion in this notebook is evidence of live-trading profitability.